In [4]:
# ============================================================
# make_figures.py
# Generates the four result figures (PDF, vector) into figures/.
# Values are the final, seed-fixed results hard-coded from the pipeline.
# Run:  python make_figures.py
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

OUT = "figures"
os.makedirs(OUT, exist_ok=True)

# Journal-friendly defaults
plt.rcParams.update({
    "font.size": 10,
    "font.family": "serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 150,
    "savefig.bbox": "tight",
})

C_ADV = "#3B6FB0"   # advantaged (blue)
C_DIS = "#C0504D"   # disadvantaged (red)

# --- add near the top, after os.makedirs(OUT, exist_ok=True) ---
def save_both(fig, name):
    """Save a figure as vector PDF and 600-dpi PNG into figures/."""
    fig.savefig(f"{OUT}/{name}.pdf")                 # vector, for print
    fig.savefig(f"{OUT}/{name}.png", dpi=600)        # raster, 600 dpi

# ------------------------------------------------------------
# Figure 1 — Nominal vs. reliable recourse across scenarios
# (Results Table: recourse rates, n=200 per group)
# ------------------------------------------------------------
def figure1():
    scenarios = ["A\n(unconstr.)", "B\n(immutability)", "C\n(guardrail)"]
    rr_adv  = [100.0, 100.0, 15.0]
    brr_adv = [6.5, 5.5, 15.0]
    rr_dis  = [100.0, 100.0, 0.0]
    brr_dis = [0.0, 0.0, 0.0]

    x = np.arange(len(scenarios)); w = 0.2
    fig, ax = plt.subplots(figsize=(6.5, 3.6))

    ax.bar(x-1.5*w, rr_adv,  w, color=C_ADV, alpha=0.35, label="Advantaged — nominal RR")
    ax.bar(x-0.5*w, brr_adv, w, color=C_ADV,             label="Advantaged — reliable RR")
    ax.bar(x+0.5*w, rr_dis,  w, color=C_DIS, alpha=0.35, label="Disadvantaged — nominal RR")
    ax.bar(x+1.5*w, brr_dis, w, color=C_DIS,             label="Disadvantaged — reliable RR")

    ax.set_xticks(x); ax.set_xticklabels(scenarios)
    ax.set_ylabel("Recourse rate (%)")
    ax.set_ylim(0, 108)
    ax.set_title("Nominal recourse conceals near-total infeasibility")
    ax.legend(fontsize=7.5, loc="upper right", framealpha=0.9)

    # annotate the key contrast
    ax.annotate("100% nominal\n→ 0% reliable", xy=(1+0.5*w, 100), xytext=(1.15, 70),
                fontsize=7.5, color=C_DIS,
                arrowprops=dict(arrowstyle="->", color=C_DIS, lw=0.8))
    save_both(fig, "fig1_scenario_tradeoff")
    plt.close(fig)
# ------------------------------------------------------------
# Figure 2 — Reliable recourse: advantaged vs disadvantaged
# with 95% intervals (non-overlapping)
# ------------------------------------------------------------
def figure2():
    groups = ["Advantaged\n(n=200)", "Disadvantaged\n(full pool, n=785)"]
    est = [15.0, 0.127]
    lo  = [10.5, 0.0]
    hi  = [20.0, 0.708]
    colors = [C_ADV, C_DIS]

    yerr = np.array([[est[i]-lo[i] for i in range(2)],
                     [hi[i]-est[i] for i in range(2)]])
    x = np.arange(2)
    fig, ax = plt.subplots(figsize=(5.2, 3.8))
    ax.bar(x, est, 0.5, color=colors, alpha=0.85,
           yerr=yerr, capsize=8, error_kw=dict(lw=1.2))

    for i in range(2):
        label = f"{est[i]:.2f}%" if est[i] < 1 else f"{est[i]:.1f}%"
        ax.text(x[i], hi[i]+0.9, label, ha="center", fontsize=9)
    ax.text(1, 3.5, "95% upper\nbound 0.71%", ha="center", fontsize=7.5, color=C_DIS)

    ax.set_xticks(x); ax.set_xticklabels(groups)
    ax.set_ylabel("Borrower-Reliable RR (%), Scenario C")
    ax.set_ylim(0, 23)
    ax.set_title("Reliable recourse: non-overlapping confidence regions")
# figure2()  마지막 줄
    save_both(fig, "fig2_reliable_rr_groups")
    plt.close(fig)

# ------------------------------------------------------------
# Figure 3 — CDR sensitivity heatmap (9 cutoff configurations)
# ------------------------------------------------------------
def figure3():
    tenure = ["p33/p67", "p25/p75", "p20/p80"]     # loose → strict (rows, bottom-up)
    age    = ["≥50", "≥60", "≥70"]                 # broad → narrow (cols)
    # CDR[tenure_row][age_col]
    cdr = np.array([
        [10.5, 13.5, 15.9],   # p33/p67
        [14.2, 17.2, 23.1],   # p25/p75
        [18.2, 20.5, 26.0],   # p20/p80
    ])
    fig, ax = plt.subplots(figsize=(5.2, 3.8))
    im = ax.imshow(cdr, cmap="Reds", aspect="auto", vmin=10, vmax=27)

    ax.set_xticks(range(3)); ax.set_xticklabels(age)
    ax.set_yticks(range(3)); ax.set_yticklabels(tenure)
    ax.set_xlabel("Old-age-band threshold")
    ax.set_ylabel("Tenure cutoff (short/long percentile)")
    ax.set_title("Approval disparity (CDR) is robust to cutoffs")

    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cdr[i,j]:.1f}×", ha="center", va="center",
                    color="white" if cdr[i,j] > 19 else "black", fontsize=10)
    cbar = fig.colorbar(im, ax=ax, shrink=0.85)
    cbar.set_label("CDR (advantaged ÷ disadvantaged)")
# figure3()  마지막 줄
    save_both(fig, "fig3_cdr_sensitivity")
    plt.close(fig)

# ------------------------------------------------------------
# Figure 4 — Double Exclusion Index decomposed into two axes
# DEI = P(reject) × P(no reliable recourse)
# ------------------------------------------------------------
def figure4():
    groups = ["Advantaged", "Disadvantaged"]
    p_reject   = [0.632, 0.979]
    p_no_rec   = [0.85, 1.00]
    dei        = [0.537, 0.979]
    x = np.arange(2)

    fig, ax = plt.subplots(figsize=(6.0, 3.8))
    # show the two multiplicative factors as light bars, DEI as the solid overlay
    ax.bar(x-0.2, p_reject, 0.35, color="#8DA0C0", label="P(rejection)")
    ax.bar(x+0.2, p_no_rec, 0.35, color="#D8A0A0", label="P(no reliable recourse)")
    ax.plot(x, dei, "ko-", lw=1.5, label="DEI = product")
    for i in range(2):
        ax.text(x[i], dei[i]+0.03, f"DEI={dei[i]:.2f}", ha="center", fontsize=9)

    ax.set_xticks(x); ax.set_xticklabels(groups)
    ax.set_ylabel("Probability")
    ax.set_ylim(0, 1.12)
    ax.set_title("Double Exclusion Index: two axes combined")
    ax.legend(fontsize=8, loc="lower right")
# figure4()  마지막 줄
    save_both(fig, "fig4_dei_decomposition")
    plt.close(fig)

if __name__ == "__main__":
    figure1(); figure2(); figure3(); figure4()
    print("Saved 4 figures to", OUT)

Saved 4 figures to figures
